# Preprocessing pipeline

Transforms the merged school-level dataset (from `school_per_EDA.ipynb`) into
model-ready arrays for three targets:

- **Ofsted grade** (classification, 4 classes) — historical study of the pre-Sept-2024 framework  
- **Progress 8 2023/24** (regression) — school-effect measure, latest year P8 was published  
- **Attainment 8 2024/25** (regression) — forward-looking predictive target, temporal split

The same feature set feeds all three; only the target and row subset differ.

## 1. Load merged dataset

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
DATA = Path("../data")

df = pd.read_csv(DATA / "merged_school_dataset.csv")
print(f"Loaded merged dataset: {df.shape[0]:,} schools × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())

Loaded merged dataset: 3,439 schools × 27 columns
Columns: ['urn', 'lsoa', 'fsm_pct_2023', 'a8_2023', 'p8_2023', 'fsm_pct_2024', 'a8_2024', 'p8_2024', 'fsm_pct_2025', 'a8_2025', 'p8_2025', 'lsoa_code_2021', 'imd_decile', 'imd_score', 'income_score', 'edu_skills_score', 'employment_score', 'health_score', 'crime_score', 'barriers_score', 'living_env_score', 'idaci_score', 'ofsted_grade', 'ofsted_grade_label', 'ofsted_num', 'p8_missing', 'grade_missing']


## 2. Target preparation

Three targets, three subsets. Different rows per target, same features.

In [2]:
# ─── Ofsted classification subset (historical study) ────────────────────
# Keep only schools with a usable graded judgement (1–4).
# NaN and "Not judged" are treated as missing targets and excluded.
graded_mask = df["ofsted_grade"].astype(str).str.strip().isin(["1", "2", "3", "4"])
df_cls = df.loc[graded_mask].copy()

# Preserve the original 1–4 scale for interpretation and reporting.
df_cls["ofsted_num"] = df_cls["ofsted_num"].astype(int)
grade_labels = ["Inadequate", "Requires improvement", "Good", "Outstanding"]

# Shift to 0-based labels for PyTorch's CrossEntropyLoss.
# 0 = Inadequate, 1 = RI, 2 = Good, 3 = Outstanding.
# Higher label = better outcome, matching the ordinal direction used in the EDA correlations.
y_cls = df_cls["ofsted_num"] - 1

# ─── Progress 8 regression subset ────────────────────────────────────────
# Uses P8 for 2023/24 — the latest year with published Progress 8.
# P8 is not published for 2024/25 because the KS2 baseline was cancelled during COVID.
df_reg_p8 = df.loc[df["p8_2024"].notna()].copy()
y_reg_p8  = df_reg_p8["p8_2024"]

# ─── Attainment 8 regression subset ──────────────────────────────────────
# A8 is published every year; used as the forward-looking predictive target.
# Temporal train/test split will be constructed in the splits section below.
df_reg_a8 = df.loc[df["a8_2025"].notna()].copy()
y_reg_a8  = df_reg_a8["a8_2025"]

print(f"Ofsted classification: {len(df_cls):,} schools")
print(f"P8 regression:         {len(df_reg_p8):,} schools")
print(f"A8 regression:         {len(df_reg_a8):,} schools")
print("\nClass distribution (Ofsted graded sample):")
print(y_cls.value_counts().sort_index().rename(index=dict(enumerate(grade_labels))))
print("\nClass proportions (%):")
print((y_cls.value_counts(normalize=True).sort_index() * 100).round(1)
      .rename(index=dict(enumerate(grade_labels))))

# Sanity checks — the auditor view.
# Ofsted count corresponds to the 6 July 2026 data snapshot (MI extract as at 31 May 2026);
# expected value may drift with fresher extracts, so a range check is used rather than strict equality.
assert y_cls.notna().all(), "classification target still has NaN"
assert y_cls.isin([0, 1, 2, 3]).all(), "unexpected class labels"
assert 1750 <= len(df_cls) <= 1900, f"Ofsted subset unusually sized: {len(df_cls):,}"

Ofsted classification: 1,814 schools
P8 regression:         3,141 schools
A8 regression:         3,215 schools

Class distribution (Ofsted graded sample):
ofsted_num
Inadequate                55
Requires improvement     251
Good                    1236
Outstanding              272
Name: count, dtype: int64

Class proportions (%):
ofsted_num
Inadequate               3.0
Requires improvement    13.8
Good                    68.1
Outstanding             15.0
Name: proportion, dtype: float64


## 3. Feature selection


1. **`edu_skills_score`** — the IMD Education, Skills and Training domain score is constructed partly from school attainment data (KS2/KS4 results feed the Children & Young People sub-domain). Using it to predict school performance is partially circular. Excluded to preserve interpretability of SHAP results.
2. **`imd_decile`** — retained for EDA plots but not used as a model input; `imd_score` carries the same information continuously and more informatively.
3. **`ofsted_grade` / `ofsted_grade_label` / `ofsted_num`** — these are the target; excluded from features to prevent target leakage.
4. **Prior-year performance** (`p8_2023`, `a8_2024`) — strong (potentially dominant) predictors. Included for now; will be ablated later to isolate the socio-economic signal.

In [3]:
# Numeric features — expand to full IMD domain set in the EDA merge, then re-list here
numeric_features = [
    "fsm_pct_2024",
    "imd_score",
    "income_score",
    "employment_score",
    "health_score",
    "crime_score",
    "barriers_score",
    "living_env_score",
    "idaci_score",
    # "edu_skills_score",  # EXCLUDED: constructed from attainment (leakage)
    "p8_2023",
    "a8_2024",
]

categorical_features = [
    # "urbanrural_name",   # add once carried through the EDA merge
]

feature_cols = numeric_features + categorical_features

# Extract feature matrices per target — same columns, different row sets.
X_cls    = df_cls[feature_cols].copy()
X_reg_p8 = df_reg_p8[feature_cols].copy()
X_reg_a8 = df_reg_a8[feature_cols].copy()

print(f"Feature count: {len(feature_cols)} "
      f"({len(numeric_features)} numeric, {len(categorical_features)} categorical)")
print(f"\nShapes:  X_cls {X_cls.shape}  |  X_reg_p8 {X_reg_p8.shape}  |  X_reg_a8 {X_reg_a8.shape}")
print("\nMissing values per feature (classification subset):")
print(X_cls.isna().sum().sort_values(ascending=False))

Feature count: 11 (11 numeric, 0 categorical)

Shapes:  X_cls (1814, 11)  |  X_reg_p8 (3141, 11)  |  X_reg_a8 (3215, 11)

Missing values per feature (classification subset):
p8_2023             209
a8_2024             154
fsm_pct_2024        154
income_score          0
imd_score             0
employment_score      0
health_score          0
barriers_score        0
crime_score           0
idaci_score           0
living_env_score      0
dtype: int64


## 4. Train / validation / test splits

Three targets, three split designs — each matched to the research question:

- **Ofsted (classification):** stratified 70/15/15 split, seeded, to preserve
  the imbalanced class proportions across all three partitions.
- **Progress 8 (regression):** random 70/15/15 split — only one year of P8
  data is available (2023/24), so no temporal design is possible.
- **Attainment 8 (regression):** temporal split — train on 2022/23 and 2023/24
  outcomes pooled, forward-test on 2024/25. Uses `GroupKFold` on URN during
  cross-validation so a school's two training-year rows always fall together.

### 4.1 Ofsted — stratified split

In [4]:
from sklearn.model_selection import train_test_split

SEED = 42

# ─── Ofsted: stratified 70/15/15 ─────────────────────────────────────────
# First split: hold out 30% (val + test combined)
X_cls_train, X_cls_temp, y_cls_train, y_cls_temp = train_test_split(
    X_cls, y_cls,
    test_size=0.30,
    stratify=y_cls,       # preserve class proportions
    random_state=SEED,
)

# Second split: divide the 30% into 15% val and 15% test
X_cls_val, X_cls_test, y_cls_val, y_cls_test = train_test_split(
    X_cls_temp, y_cls_temp,
    test_size=0.50,       # half of 30% = 15% each
    stratify=y_cls_temp,
    random_state=SEED,
)

# Verify sizes and class proportions preserved across splits
print(f"Ofsted split sizes:")
print(f"  Train: {len(X_cls_train):>5,}  ({len(X_cls_train)/len(X_cls):.1%})")
print(f"  Val:   {len(X_cls_val):>5,}  ({len(X_cls_val)/len(X_cls):.1%})")
print(f"  Test:  {len(X_cls_test):>5,}  ({len(X_cls_test)/len(X_cls):.1%})")

print(f"\nClass proportions (%) across splits — should be near-identical:")
proportions = pd.DataFrame({
    "train": y_cls_train.value_counts(normalize=True).sort_index() * 100,
    "val":   y_cls_val.value_counts(normalize=True).sort_index() * 100,
    "test":  y_cls_test.value_counts(normalize=True).sort_index() * 100,
}).round(1)
proportions.index = grade_labels
print(proportions)

# Sanity checks
assert len(X_cls_train) + len(X_cls_val) + len(X_cls_test) == len(X_cls)
assert set(y_cls_train.unique()) == set(y_cls.unique()), "class missing from train"

Ofsted split sizes:
  Train: 1,269  (70.0%)
  Val:     272  (15.0%)
  Test:    273  (15.0%)

Class proportions (%) across splits — should be near-identical:
                      train   val  test
Inadequate              3.0   3.3   2.9
Requires improvement   13.9  13.6  13.9
Good                   68.2  68.0  68.1
Outstanding            15.0  15.1  15.0


### 4.2 Progress 8 — random split

P8 for 2023/24 is only available as a single year, so no temporal design is
possible. Uses a random 70/15/15 split, seeded for reproducibility.


In [5]:
# ─── Progress 8: random 70/15/15 ─────────────────────────────────────────
X_reg_p8_train, X_reg_p8_temp, y_reg_p8_train, y_reg_p8_temp = train_test_split(
    X_reg_p8, y_reg_p8,
    test_size=0.30,
    random_state=SEED,
)

X_reg_p8_val, X_reg_p8_test, y_reg_p8_val, y_reg_p8_test = train_test_split(
    X_reg_p8_temp, y_reg_p8_temp,
    test_size=0.50,       # half of 30% = 15% each
    random_state=SEED,
)

# Verify sizes and target distribution across splits
print(f"P8 split sizes:")
print(f"  Train: {len(X_reg_p8_train):>5,}  ({len(X_reg_p8_train)/len(X_reg_p8):.1%})")
print(f"  Val:   {len(X_reg_p8_val):>5,}  ({len(X_reg_p8_val)/len(X_reg_p8):.1%})")
print(f"  Test:  {len(X_reg_p8_test):>5,}  ({len(X_reg_p8_test)/len(X_reg_p8):.1%})")

print(f"\nP8 target distribution — should be near-identical across splits:")
distribution = pd.DataFrame({
    "train": y_reg_p8_train.describe(),
    "val":   y_reg_p8_val.describe(),
    "test":  y_reg_p8_test.describe(),
}).round(3)
print(distribution)

# Sanity checks
assert len(X_reg_p8_train) + len(X_reg_p8_val) + len(X_reg_p8_test) == len(X_reg_p8)

P8 split sizes:
  Train: 2,198  (70.0%)
  Val:     471  (15.0%)
  Test:    472  (15.0%)

P8 target distribution — should be near-identical across splits:
          train      val     test
count  2198.000  471.000  472.000
mean      0.003   -0.015    0.006
std       0.508    0.524    0.571
min      -1.940   -1.900   -3.840
25%      -0.340   -0.340   -0.312
50%       0.000   -0.040    0.000
75%       0.340    0.290    0.352
max       2.560    1.540    2.090
